# ODI to Databricks Migration: SCEN_TASK_NO

**Source File:** `SCEN_TASK_NO`
**Conversion Timestamp:** 2024-07-30T12:00:00Z
**Description:** This notebook converts an ODI scenario with a simple insert operation into the `TRG_LOC` table, sourcing data from `LOCATIONS`.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "-1", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "-1", "ETL Process Widget ID")
dbutils.widgets.text("ODI_SESS_NO", "-1", "ODI Session Number")

## ETL Parameters

In [ ]:
%sql
SELECT
  '${ETL_JOB_TYPE}' AS ETL_JOB_TYPE,
  '${DATASOURCE_NUM_ID}' AS DATASOURCE_NUM_ID,
  '${ETL_PROC_WID}' AS ETL_PROC_WID,
  '${ODI_SESS_NO}' AS ODI_SESS_NO;

## Target Table Creation

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS workspace.hr.trg_loc (
    LOCATION_ID     BIGINT,
    STREET_ADDRESS  STRING,
    POSTAL_CODE     STRING,
    CITY            STRING,
    STATE_PROVINCE  STRING,
    COUNTRY_ID      STRING
) USING DELTA;

## SCEN_TASK_NO {30}: Insert into Target Table

In [ ]:
%sql
-- SCEN_TASK_NO {10}: Initial task, no SQL provided in source.
-- SCEN_TASK_NO {20}: Intermediate task, no SQL provided in source.
INSERT INTO workspace.hr.trg_loc
(
    LOCATION_ID,
    STREET_ADDRESS,
    POSTAL_CODE,
    CITY,
    STATE_PROVINCE,
    COUNTRY_ID
)
SELECT
    LOCATIONS.LOCATION_ID,
    LOCATIONS.STREET_ADDRESS,
    LOCATIONS.POSTAL_CODE,
    LOCATIONS.CITY,
    LOCATIONS.STATE_PROVINCE,
    LOCATIONS.COUNTRY_ID
FROM
    workspace.hr.locations AS LOCATIONS;

## Validation

In [ ]:
%sql
SELECT COUNT(*) AS total_rows FROM workspace.hr.trg_loc;

## Conversion Notes

- **Schema and Table Naming:** Original Oracle schema `HR` converted to `workspace.hr`. Table names `TRG_LOC` and `LOCATIONS` converted to lowercase `trg_loc` and `locations` respectively.
- **Oracle Hints:** The `/*+ APPEND PARALLEL */` hint has been removed as it is specific to Oracle and not applicable in Databricks Spark SQL.
- **Task Handling:** `SCEN_TASK_NO {10}` and `SCEN_TASK_NO {20}` had no associated SQL in the source and are preserved as comments within the adjacent SQL cell for `SCEN_TASK_NO {30}`.
- **DDL Inference:** A `CREATE TABLE IF NOT EXISTS` statement for `workspace.hr.trg_loc` has been added with inferred data types (e.g., `NUMBER(p,0)` to `BIGINT`, `VARCHAR2` to `STRING`) to ensure the target table exists before the `INSERT` operation. This DDL was not present in the original ODI source.
- **No Parameters Used:** Although standard ETL parameters are created as widgets, they are not directly referenced in this specific SQL due to the simplicity of the source query.